In [4]:
import os
import time
import requests
from bs4 import BeautifulSoup
from git import Repo  # Requires 'gitpython' package

# ==================== CONFIGURATION ====================
OUTPUT_HTML_FILE = "index.html"  # This will be your website's home page

TRACKING_LIST = {
    "buydeem_g553": {
        "name": "BUYDEEM G553 Food Steamer",
        "url": "https://us.buydeem.com/products/electric-food-steamer-g553",
        "baseline_price": 229.99
    },
    "example_item_2": {
        "name": "Alternative Multi-Cooker",
        "url": "https://example.com",
        "baseline_price": 150.00
    }
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}
# ========================================================

def fetch_price(url):
    """Fetches the price from a given e-commerce URL."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Strategy 1: Shopify / OpenGraph Price Meta
        price_meta = soup.find("meta", property="og:price:amount") or soup.find("meta", name="twitter:data1")
        if price_meta and price_meta.get("content"):
            return float(price_meta["content"].replace("$", "").strip())

        # Strategy 2: Common HTML Class Fallback
        price_elements = soup.find_all(class_=lambda x: x and 'price' in x.lower())
        for elem in price_elements:
            text = elem.get_text().strip()
            if "$" in text:
                import re
                match = re.search(r'\d+\.\d{2}', text)
                if match:
                    return float(match.group())
        return None
    except Exception:
        return None

def push_to_github():
    """Pushes the updated index.html file to your public GitHub repo."""
    try:
        repo_path = os.getcwd()
        repo = Repo(repo_path)
        
        # Stage everything (including this script if you modified it), commit, and push
        repo.git.add(A=True)
        repo.index.commit(f"Manual price update: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        origin = repo.remote(name='origin')
        origin.push()
        print("🌍 Success! Dashboard pushed live to GitHub Pages.")
    except Exception as e:
        print(f"⚠️ Git Push Failed: {e}")
        print("Your local file was still saved! You can double-click index.html to view it locally.")

def run_single_check():
    """Runs a single check, writes the HTML, and pushes it online."""
    print("🚀 Booting tracker... Fetching live prices right now...")
    report_items = []
    any_drops = False

    for key, item in TRACKING_LIST.items():
        current_price = fetch_price(item["url"])
        baseline = item["baseline_price"]
        
        if current_price is None:
            status_td = "<td style='color: #7f8c8d; font-weight: bold;'>Error fetching price</td>"
        elif current_price < baseline:
            any_drops = True
            savings = baseline - current_price
            status_td = f"<td style='color: #27ae60; background-color: #e8f8f5; font-weight: bold;'>🚨 PRICE DROP: ${current_price:.2f} (Saved ${savings:.2f})</td>"
        else:
            status_td = f"<td style='color: #2c3e50;'>Normal (${current_price:.2f})</td>"
            
        report_items.append(f"""
            <tr>
                <td><b>{item['name']}</b></td>
                <td>${baseline:.2f}</td>
                {status_td}
                <td><a href='{item['url']}' target='_blank'>Shop Link ↗</a></td>
            </tr>
        """)

    # Generate HTML code string
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Price Tracker Dashboard</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background-color: #f8f9fa; margin: 40px; color: #333; }}
            .container {{ max-width: 900px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}
            h1 {{ margin-top: 0; color: #2c3e50; font-size: 24px; }}
            .timestamp {{ color: #7f8c8d; font-size: 14px; margin-bottom: 20px; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 10px; }}
            th, td {{ padding: 12px 15px; text-align: left; border-bottom: 1px solid #e1e8ed; }}
            th {{ background-color: #f4f6f8; color: #5c7080; font-weight: 600; }}
            a {{ color: #3498db; text-decoration: none; font-weight: 500; }}
            a:hover {{ text-decoration: underline; }}
            .alert-banner {{ background-color: #e74c3c; color: white; padding: 15px; border-radius: 6px; font-weight: bold; margin-bottom: 20px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>🛍️ Price Monitor Dashboard</h1>
            <div class="timestamp">Last Updated: {time.strftime('%Y-%m-%d %I:%M:%S %p')}</div>
            
            { '<div class="alert-banner">🔥 Deals Found! One or more tracked items have dropped below your baseline price.</div>' if any_drops else '' }
            
            <table>
                <thead>
                    <tr>
                        <th>Product Name</th>
                        <th>Target Baseline</th>
                        <th>Current Status</th>
                        <th>Action</th>
                    </tr>
                </thead>
                <tbody>
                    {"".join(report_items)}
                </tbody>
            </table>
        </div>
    </body>
    </html>
    """

    # Write file locally
    with open(OUTPUT_HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"✅ Local {OUTPUT_HTML_FILE} file successfully built.")
    
    # Upload everything live
    push_to_github()

if __name__ == "__main__":
    run_single_check()
    print("🏁 Run complete. Script closed down safely.")


🚀 Booting tracker... Fetching live prices right now...
✅ Local index.html file successfully built.
🌍 Success! Dashboard pushed live to GitHub Pages.
🏁 Run complete. Script closed down safely.
